---
title: Introduction to SpotOptim
sidebar_position: 1
eval: true
---

# Examples

Let us consider the problem of minimizing the Rosenbrock function. This function (and its respective derivatives) is implemented in `rosen` (and `rosen_der`, `rosen_hess`) in the `scipy.optimize` module.

The function is usually evaluated on the hypercube $x_i \in [-5, 10]$, for all $i = 1, \ldots, d$, although it may be restricted to the hypercube $x_i \in [-2.048, 2.048], for all $i = 1, \ldots, d$, see [https://www.sfu.ca/~ssurjano/rosen.html](https://www.sfu.ca/~ssurjano/rosen.html).

A simple application of the Nelder-Mead method is:

In [ ]:
from scipy.optimize import minimize, rosen, rosen_der
x0 = [1.3, 0.7, 0.8, 1.9, 1.2]
res = minimize(rosen, x0, method='Nelder-Mead', tol=1e-6, 
        bounds = [(-2.048, 2.048)] * 5)
res.x

Now using the BFGS algorithm, using the first derivative and a few options:

In [ ]:
res = minimize(rosen, x0, method='BFGS', jac=rosen_der,
               options={'gtol': 1e-6, 'disp': True})
res.x
print(res.message)

Now, let's see how to solve the same problem using **SpotOptim**, which uses a Surrogate-Model Based Optimization (SMBO) approach. Unlike `minimize`, SpotOptim requires bounds as it samples the search space globally.

In [ ]:
import numpy as np
from spotoptim import SpotOptim
from scipy.optimize import rosen

# SpotOptim expects function input as a 2D array (batch of points)
# So we wrap the rosen function to handle (n_samples, n_dim) input
def rosen_batch(X):
    return np.array([rosen(x) for x in X])

# Define the optimizer
# We use a 5-dimensional problem similar to the example above
# Bounds are required for SpotOptim
optimizer = SpotOptim(
    fun=rosen_batch,
    bounds = [(-2.048, 2.048)] * 5,
    max_iter=50,  # Total budget of function evaluations
    n_initial=10, # Initial random samples
    seed=42
)

# Run optimization
res = optimizer.optimize()
print(res.message)

In [ ]:
print(f"Best solution found: {res.x}")
print(f"Best objective value: {res.fun}")
print(f"Total evaluations: {res.nfev}")

SpotOptim is particularly useful when the objective function is expensive to evaluate (e.g., simulations, hyperparameter tuning), as it builds a model to intelligently select the next point to evaluate, often finding good solutions with fewer function calls than gradient-free local search methods.

## Complex Constrained Optimization: The Robot Arm

For a more challenging example, let's consider the `robot_arm_hard` problem. This is a 10-dimensional problem where a simulated robot arm must reach a target while avoiding obstacles in a maze-like configuration. It features "hard" constraints implemented as severe penalties, creating a rugged landscape that is difficult for local optimizers.

First, let's try solving it with **Scipy's** default local optimizer. Note that we need to wrap the function to return a scalar float, as `robot_arm_hard` returns an array.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from spotoptim.function.so import robot_arm_hard

# Wrapper for Scipy (expects scalar return)
def objective_scipy(x):
    # robot_arm_hard handles constraints internally via penalties
    return float(robot_arm_hard(x)[0])

# Starting point
np.random.seed(42)
x0 = np.random.rand(10)
print(f"Initial cost: {objective_scipy(x0):.4f}")

# Run generic minimization (Nelder-Mead is default for gradient-free)
res_scipy = minimize(objective_scipy, x0, method='Nelder-Mead', tol=1e-4,
            bounds = [(0.0, 1.0)] * 10)

print(f"Scipy Best cost: {res_scipy.fun:.4f}")
print(f"Scipy Success: {res_scipy.success}")

The local optimizer often gets stuck in local optima created by the obstacles (penalties), failing to find a path to the target.

Now let's use **SpotOptim**. We don't need a wrapper because SpotOptim natively supports the batch-vectorized output of `robot_arm_hard`. We simply define the bounds for the 10 joint angles.

In [ ]:
from spotoptim import SpotOptim

# Define bounds: 10 angles normalized to [0, 1]
bounds = [(0.0, 1.0)] * 10

optimizer = SpotOptim(
    fun=robot_arm_hard,  # Native array-based function
    bounds=bounds,
    max_iter=50,         # Budget
    n_initial=20,        # More exploration for complex space
    seed=42,
    max_surrogate_points=30
)

res_spot = optimizer.optimize()

print(f"SpotOptim Best cost: {res_spot.fun:.4f}")

SpotOptim's use of a surrogate model and global acquisition function allows it to better explore the landscape and often jump out of the local traps that catch the local optimizer, finding a significantly better configuration for the robot arm.

## Jupyter Notebook

:::{.callout-note}

* The Jupyter-Notebook of this chapter is available on GitHub in the [Sequential Parameter Optimization Cookbook Repository](https://github.com/sequential-parameter-optimization/spotoptim-cookbook/blob/main/spotoptim_intro.ipynb)

:::